<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **12. Understanding Inference Control**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 12.1 Introduction

In [05.05_Introduction_to_LLMs.ipynb](05.05_Introduction_to_LLMs.ipynb) you reshaped the probabilities of a single next token by hand: temperature, top-k, top-p. A real answer is a chain of such draws, and this notebook follows the same settings from one token to whole answers, live on the Pi. It also covers the settings that decide where an answer ends, how much of your prompt the model reads at all, and what form the answer takes.

None of this is specific to Ollama. The APIs of OpenAI, Anthropic and Google offer the same controls under similar names: temperature, top-p, a token limit, stop sequences, a seed and a response format. What you learn here carries over to every model you will call from a program.

Allow about **60 minutes**, plus time for the questions.

By the end, you should be able to predict how temperature, top-k and top-p change a model's answers, make a run repeatable, end an answer where you want, notice when a prompt no longer fits the context window, and get output that a program can read.

#### 12.1.1 Environment Setup

Everything runs on your Pi. Most cells use `qwen3.5:0.8b`, the smallest model of the ladder in 05.07: this notebook asks many short questions, and speed matters more here than wisdom.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

In [ ]:
import collections
import json
import random

import numpy as np

import caspi

caspi.connect()
MODEL = "qwen3.5:0.8b"
caspi.load(MODEL)

### 12.2 From One Token to Whole Answers

We ask the model for the name of a Swiss town. First, the probabilities of its first token, as in 05.05:

In [ ]:
TOWN = "Name one town in Switzerland. Answer with the name only."

reply = caspi.ask(MODEL, TOWN, think=False, num_predict=1, logprobs=True, top_logprobs=8,
                  show=False)
for candidate in reply.logprobs[0]["top_logprobs"]:
    print(f"{candidate['token']!r:>10}  {np.exp(candidate['logprob']):.3f}")

No candidate is much ahead of the others, and several are only the start of a name: `'Bas'` becomes Basel and `'Gene'` Geneva in the next step. The answer is decided token by token, and every token is one more draw.

So we ask ten times at three temperatures and count the answers. Each of the ten runs gets its own seed (Section 12.3 explains why). A question mark after a name means it is not on our short list of Swiss towns: check those yourself.

In [ ]:
SWISS = {"Zurich", "Bern", "Basel", "Geneva", "Lausanne", "Lucerne", "Lugano", "Winterthur", "St. Gallen",
         "Biel", "Thun", "Chur", "Fribourg", "Neuchâtel", "Sion", "Zug", "Aarau", "Olten", "Locarno",
         "Bellinzona", "Montreux", "Davos", "Zermatt", "Interlaken", "Grindelwald", "St. Moritz", "Baden",
         "Solothurn", "Schaffhausen", "Vevey", "Nyon", "Ascona", "Appenzell", "Einsiedeln", "Lenzburg", "Brig"}
SPELLINGS = {"Zürich": "Zurich", "Genf": "Geneva", "Genève": "Geneva", "Berne": "Bern", "Luzern": "Lucerne",
             "Lucerna": "Lucerne", "Sankt Gallen": "St. Gallen", "Freiburg": "Fribourg"}


def answer(prompt, **settings):
    """One answer as plain text, without printing and without thinking."""
    return caspi.ask(MODEL, prompt, think=False, show=False, **settings).text.strip(" .")


def sample(prompt, n=10, **settings):
    """Ask n times, with the seeds 0 to n-1, and count the answers. Spellings of one town count once."""
    answers = [answer(prompt, seed=seed, num_predict=8, **settings) for seed in range(n)]
    return collections.Counter(SPELLINGS.get(a, a) for a in answers)


def towns(counts):
    return ", ".join(f"{town} {n}" + ("" if town in SWISS else " ?") for town, n in counts.most_common())


for t in (0.1, 1.0, 1.8):
    print(f"temperature {t}:  {towns(sample(TOWN, temperature=t))}")

At a low temperature the model sticks to the few towns at the top of its list. As the temperature rises, the answers spread out, and at some point names appear that are no towns at all. Look at how they are built: every piece is a likely token, only the combination does not exist. This is how language models invent things, and [05.13_Experimenting_with_Prompts.ipynb](05.13_Experimenting_with_Prompts.ipynb) comes back to it.

<div class="alert alert-block alert-info">

Predict before you run the next cell: at temperature 1.8, which of the two settings removes the invented towns, `top_k=5` or `top_p=0.5`? Keep in mind what each one keeps at every step: `top_k` a fixed number of candidates, `top_p` as many as it takes to reach the probability you give, which depends on how flat the distribution is.
</div>

In [ ]:
for settings in ({"top_k": 5}, {"top_p": 0.5}):
    print(f"temperature 1.8, {settings}:  {towns(sample(TOWN, temperature=1.8, **settings))}")

Both settings cut the long tail of unlikely tokens, where the invented names come from, and the temperature still mixes what is left. Top-k sampling [01] and top-p, also called nucleus sampling [02], were introduced for this purpose: to keep sampling varied without letting it drift into nonsense. Remember that this happens at every token: after `'Bas'`, top-k and top-p decide again.

<div class="alert alert-block alert-info">

Find settings that give at least five different towns in ten answers, all of them real. Change the values in the cell and run it until it says so. Spellings of the same town count once. If a real town is marked with a question mark, add it to `SWISS`: you are the judge here.
</div>

In [ ]:
counts = sample(TOWN, temperature=0.1)   # change these settings
real = [town for town in counts if town in SWISS]
print(towns(counts))
print("solved" if len(real) >= 5 and len(real) == len(counts)
      else f"not yet: {len(real)} different real towns, {len(counts) - len(real)} unknown")

### 12.3 Same Seed, Same Answer

Sampling needs random numbers, and a seed fixes them. With the same model, prompt, settings and seed, the Pi gives the same answer every time:

In [ ]:
print("seed 7, three times: ", [answer(TOWN, temperature=1.5, seed=7, num_predict=8) for _ in range(3)])
print("no seed, three times:", [answer(TOWN, temperature=1.5, num_predict=8) for _ in range(3)])

That is what you want for debugging and for fair comparisons: change one setting, keep the seed, and any difference comes from the setting. It also explains a trap. A fixed seed makes an answer repeatable, not right. A lucky seed can make a bad setting look good. Cloud APIs offer a seed too, but most of them promise only best-effort reproducibility, because your request shares hardware with others.

### 12.4 Every Model Has Its Own Defaults

Whatever you do not set, the model's defaults decide. They come with the model, and they differ. This cell asks every model on your Pi:

In [ ]:
for model in caspi.models():
    defaults = caspi.ollama("/api/show", {"model": model}).get("parameters", "")
    print(f"{model:30s}", " | ".join(" ".join(line.split()) for line in defaults.splitlines()) or "none")

The differences are large. `lfm2.5` comes with a temperature of 0.2, the `qwen3.5` models with 1, a `top_k` of 20 and a penalty for tokens that already appeared. A model that lists none gets Ollama's own defaults: temperature 0.8, `top_k` 40, `top_p` 0.9 (see the [parameter table](https://docs.ollama.com/modelfile#valid-parameters-and-values)).

So "the same prompt with default settings" compares two different decoding rules, not just two models. Whenever you compare models, set the settings that matter yourself.

### 12.5 Where an Answer Ends

An answer ends in one of three ways, and Ollama tells you which in `done_reason`:

In [ ]:
DAYS = "List the days of the week, one per line."

for label, settings in [("its own end", {}), ("num_predict=6", {"num_predict": 6}),
                        ('stop=["Thursday"]', {"stop": ["Thursday"]})]:
    reply = caspi.ask(MODEL, DAYS, think=False, temperature=0, show=False, **settings)
    print(f"{label:18s} done_reason {reply.raw['done_reason']:7s} {' '.join(reply.text.split())}")

- **Its own end:** the model wrote its end-of-turn token. It learned to do that during training, and [05.13](05.13_Experimenting_with_Prompts.ipynb) shows what happens when it does not.
- **`num_predict`:** a hard limit in tokens, not words. It cuts wherever it is, and `done_reason` says `length`.
- **`stop`:** the answer ends cleanly before a string you choose, which is not included. The string must match exactly, capital letters included. `done_reason` says `stop`, the same as for the model's own end.

`num_predict` is a safety net. A stop sequence is a tool: it lets you end an answer at a point that makes sense for your program.

### 12.6 The Silent Context Limit

The context window, `num_ctx`, is how many tokens the model can read at once: your prompt and its answer together. On your Pi it is 8192 tokens unless you ask for more or less. [05.09_Making_Models_Smaller.ipynb](05.09_Making_Models_Smaller.ipynb) showed what a larger window costs in memory. This section shows what a too small one costs.

Here is a log of a café Wi-Fi router, with a note from the owner in its first line, and a question at the end:

In [ ]:
DEVICES = ["laptop", "phone", "card reader", "tablet", "printer", "smart TV"]
random.seed(1)
lines = ["Note from the owner: the guest Wi-Fi password is alpenglow42."]
for i in range(150):
    lines.append(f"12:{i // 60:02d}:{i % 60:02d} {random.choice(DEVICES)} joined on channel "
                 f"{random.choice([1, 6, 11])}, signal {random.randint(-80, -40)} dBm")
LOG = "\n".join(lines) + "\n\nWhat is the guest Wi-Fi password? Answer with the password only."
print("\n".join(lines[:4]), "\n...")

<div class="alert alert-block alert-info">

The log is about 3500 tokens long. Predict the answer with a window of 4096 tokens, and with one of 2048. Then run the cell. It reads the whole log twice, so give it a minute.
</div>

In [ ]:
for num_ctx in (4096, 2048):
    reply = caspi.ask(MODEL, LOG, num_ctx=num_ctx, temperature=0, num_predict=20, think=False,
                      show=False)
    print(f"num_ctx {num_ctx}: the model read {reply.raw['prompt_eval_count']} tokens and answered "
          f"{reply.text.strip()!r}")

With the smaller window, Ollama cut the prompt from the front so that it fits, and the model read only the end of the log. The password never reached it, and it answered anyway. There was no error and no warning.

This is one of the most common reasons why a model "forgets" what it was told: a long chat history, a large document or the results of many tool calls simply no longer fit. Even when a prompt fits, models tend to use its middle less reliably than its beginning and end [03]. For a program, a loud failure is better than a quiet wrong answer, and `truncate=False` asks for exactly that:

In [ ]:
try:
    caspi.ask(MODEL, LOG, num_ctx=2048, truncate=False, show=False)
except caspi.PiError as error:
    print(error)

### 12.7 Answers a Program Can Read

A program that uses a model's answer needs it in a fixed form. Asking for JSON in the prompt is a start:

In [ ]:
REPORT = ("Since Monday the card reader by the window loses Wi-Fi whenever the microwave runs. "
          "Customers cannot pay at lunch.")

free = answer("Extract device, problem and urgency from this report as JSON: " + REPORT, num_predict=150)
print(free)
try:
    json.loads(free)
except json.JSONDecodeError as error:
    print(error)

Even when the text happens to be valid JSON, nothing fixes the names of the keys or the values. A JSON schema does. Ollama turns it into a grammar [04] and, at every step, removes the tokens that would break it: their probability becomes zero, just as top-k sets the unlikely ones to zero in 05.05. The model can only choose among the tokens that fit.

In [ ]:
SCHEMA = {"type": "object", "required": ["device", "problem", "urgent"], "properties": {
    "device": {"type": "string"},
    "problem": {"enum": ["interference", "coverage", "configuration", "hardware"]},
    "urgent": {"type": "boolean"}}}

data = json.loads(answer("Extract device, problem and urgency from this report: " + REPORT,
                         format=SCHEMA, num_predict=150))
data

The form is now guaranteed: the keys are there, `problem` is one of the four words, `urgent` is `true` or `false`. The content is not. Check the problem the model picked: is it what the report describes? A schema decides what an answer can look like, never whether it is right.

<div class="alert alert-block alert-info">

Extend the schema with a field of your own, for example the Wi-Fi channel as an integer or the time the problem started. Write a second report, run the cell above with both, and check which fields the model fills correctly.
</div>

### 12.8 Summarizing Remarks

Everything you set in this notebook works on the probabilities from 05.05. Temperature, top-k and top-p decide how adventurous each draw is. A seed makes a run repeatable, not correct. `num_predict` and `stop` decide where an answer ends. The context window decides what the model reads at all, and it fails silently. A schema fixes the form of an answer, not its truth.

The same settings exist in every cloud API, often under the same names. The difference on the Pi is that you can watch each one at work.

<div class="alert alert-block alert-info">

**Try the other models.** Everything here works with every chat model on your Pi (`caspi.models()` lists them). Set `MODEL` to another one and run the notebook again. Larger models take a few minutes to load the first time.

If you want to go one step further, open a terminal, log in with `ssh cas@172.31.250.1`, and talk to Ollama's HTTP API directly. That is what caspi does for you:

```bash
curl http://localhost:11434/api/generate -d '{"model": "qwen3.5:0.8b", "prompt": "Name a Swiss town.", "stream": false, "think": false, "options": {"temperature": 1.5, "seed": 7}}'
```

The [Ollama API documentation](https://docs.ollama.com/api) lists every endpoint and option.
</div>

### 12.9 Further Reading and Literature

[01] Fan, A., Lewis, M., & Dauphin, Y. (2018). **Hierarchical Neural Story Generation.** ACL. [Paper](https://arxiv.org/abs/1805.04833). Where top-k sampling comes from.

[02] Holtzman, A., Buys, J., Du, L., Forbes, M., & Choi, Y. (2020). **The Curious Case of Neural Text Degeneration.** ICLR. [Paper](https://arxiv.org/abs/1904.09751). Introduces top-p, or nucleus, sampling.

[03] Liu, N. F., Lin, K., Hewitt, J., Paranjape, A., Bevilacqua, M., Petroni, F., & Liang, P. (2024). **Lost in the Middle: How Language Models Use Long Contexts.** TACL. [Paper](https://arxiv.org/abs/2307.03172)

[04] Willard, B. T., & Louf, R. (2023). **Efficient Guided Generation for Large Language Models.** [Paper](https://arxiv.org/abs/2307.09702). How a schema becomes a grammar that removes tokens.

[05] Ollama. **API documentation.** [Documentation](https://docs.ollama.com/api)

---
### 12.10 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: One token, whole answers**

Report your three temperature lines from Section 12.2. How does what you saw follow from the first-token list and from what you did in 05.05?

(answer here)

**Question 02: Invented towns**

Which invented names did you get? Each of their tokens was likely. Why can the combination still be a town that does not exist?

(answer here)

**Question 03: Guard rails**

Which of `top_k=5` and `top_p=0.5` kept the answers real, and was your prediction right? Paste the settings that solved the five-towns task.

(answer here)

**Question 04: Seeds**

When is a fixed seed useful? Why does it not make an answer more correct?

(answer here)

**Question 05: Defaults**

Which defaults differ most between the models on your Pi? What goes wrong if you compare two models with their default settings?

(answer here)

**Question 06: Where an answer ends**

Report the `done_reason` of each of the three runs and explain it. Why can `done_reason` not tell a stop sequence from the model's own end?

(answer here)

**Question 07: The silent limit**

What did the model answer with the smaller window, and why? Why is `truncate=False` the better choice in a program?

(answer here)

**Question 08: Form, not truth**

Paste your extended schema and the two answers. What does a schema guarantee, and what not?

(answer here)

**Question 09: Your settings**

Choose settings for a factual answer, a product slogan and a JSON extraction, and justify each with something you saw in this notebook.

(answer here)

**Question 10: Optional: cloud APIs**

Find temperature, top-p, a token limit, stop sequences and a JSON format in the documentation of one cloud LLM API. What is named differently, and what is missing?

(answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)